# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset metadata and structure are provided in Croissant format via the following URL:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

This notebook guides you step-by-step: loading metadata and record sets, reviewing schema, selecting data, performing exploratory analysis, and visualizing results. All identifiers (record set, field, column) are referenced **by their `@id`** as per Croissant best practices.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect high-level details using the `mlcroissant` package.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print main metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review all available record sets, their fields, and entity `@id`s defined in the Croissant metadata. This is critical for referencing data in future extractions and analysis.

In [ ]:
# List all record sets with their @id and fields
from pprint import pprint

if not hasattr(dataset.metadata, 'record_sets'):
    print("No record sets found in metadata.")
else:
    print("Record Sets and their Fields:\n")
    for recset in dataset.metadata.record_sets:
        print(f"@id: {recset['@id']}")
        print(f"  Name: {recset.get('name', '(unnamed)')}")
        print("  Fields:")
        for field in recset.get('fields', []):
            print(f"    @id: {field['@id']} - Name: {field.get('name', field['@id'])} - Type: {field.get('dataType', 'N/A')}")
        print()

## 3. Data Extraction
We now extract the main tabular record set(s) into Pandas DataFrames for further analysis. 
Use the `@id` of the desired record set as shown above. Most Croissant datasets only have one or two main record sets (typically the survey or primary table).

If you already know the relevant `@id` (e.g., 'cr:RecordSet/MentalHealthSurvey'), you can set it below; otherwise, use the cell above to list all options.

In [ ]:
# Set the list of record set @id(s) to extract records from
# Inspect the result of the previous cell and set accordingly

# Example: Change this list to actual recordSet @ids from previous cell's output
record_sets = [
    'cr:RecordSet/MainSurvey'  # Replace with actual @id per your dataset
]

# For demonstration: extract data
dataframes = dict()
for record_set in record_sets:
    print(f"Loading data for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} records. Fields: {list(df.columns)}\n")
# Display first few records for the main record set
main_rs = record_sets[0]
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)

Let's illustrate basic analysis: filtering, normalizing a numerical field (e.g., total score), and grouping by another field such as gender or education. Ensure to use the `@id` for all columns/fields.

In [ ]:
# === Set field @id variables for analysis ===

# Example: actual field @ids from your schema (adjust field names to match your dataset's)
numeric_field = 'cr:field/gad7_total_score'  # E.g., GAD-7 total score field
group_field = 'cr:field/gender'             # E.g., Gender field

df = dataframes[main_rs].copy()

# If necessary, cast to numeric (Croissant doesn't always enforce types)
if numeric_field in df:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):\n")
display(filtered_df[[numeric_field, group_field]].head())

# Normalize
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print("\nNormalized values (z-score):")
display(filtered_df[[numeric_field, numeric_field + '_normalized', group_field]].head())

# Group by another field if it exists
if group_field in filtered_df.columns:
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field]
        .mean()
        .reset_index()
        .rename(columns={numeric_field: numeric_field+'_mean'})
    )
    print(f"\nMean of {numeric_field} grouped by {group_field}:")
    display(grouped_df)
else:
    print(f"Group field {group_field} not found in columns.")

## 5. Visualization

Let's plot the distribution of a numeric score, and mean score by group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
else:
    print(f"Field {numeric_field} not available for plotting.")

if group_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()
else:
    print(f"Field {group_field} not available for grouping.")

## 6. Conclusion

- Demonstrated how to reference data using Croissant `@id`s for tables and fields.
- Loaded the main survey data, filtered, normalized, and grouped by demographic features using Pandas.
- Plotted distributions and groupwise summaries to gain initial insight into the mental health scores surveyed in Kilifi County.

To go further:
- Consult the dataset schema for more fields or tables to analyze.
- Apply more advanced machine learning workflows—Croissant supports reproducible, FAIR data-driven pipelines.
- Respect the dataset's limitations and privacy guidelines, as detailed in the metadata.